In [0]:
spark

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog", "consumer_goods", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
base_path = f'/Volumes/consumer_goods/files/input_data_files/child_company/{data_source}'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"

print(catalog)
print(data_source)
print(base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

#pre-defining tables 
bronze_table = f"{catalog}.bronze.{data_source}"
silver_table = f"{catalog}.silver.{data_source}"
gold_table = f"{catalog}.gold.sb_fact_{data_source}"


consumer_goods
orders
/Volumes/consumer_goods/files/input_data_files/child_company/orders
Landing Path:  /Volumes/consumer_goods/files/input_data_files/child_company/orders/landing/
Processed Path:  /Volumes/consumer_goods/files/input_data_files/child_company/orders/processed/


Most of the transformations are same as full load

In [0]:
df = (spark.read.format("csv")
        .option("header", "True")
        .option("inferSchema", "True")
        .load(f"{landing_path}/*.csv")
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*","_metadata.file_name")
)
print("Total Rows: ", df.count())
df.show(10)

Total Rows:  1712
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FDEC85903601|Thursday, Decembe...|     789903|  77777777|    342.0|2026-03-04 12:35:...|orders_2025_12_04...|
|FDEC85903601|Thursday, Decembe...|     789903|  25891302|     53.0|2026-03-04 12:35:...|orders_2025_12_04...|
|FDEC85903601|Thursday, Decembe...|     789903|  77777777|    470.0|2026-03-04 12:35:...|orders_2025_12_04...|
|FDEC85903601|Thursday, Decembe...|     789903|  25891601|    184.0|2026-03-04 12:35:...|orders_2025_12_04...|
|FDEC85903601|Thursday, Decembe...|     789903|  25891101|    459.0|2026-03-04 12:35:...|orders_2025_12_04...|
|FDEC85903601|Thursday, Decembe...|     ABC987|  25891202|    275.0|2026-03-04 12:35:...|order

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(bronze_table)

Staging table only for current data

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.bronze.staging_{data_source}")

In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(file_info.path,f"{processed_path}/{file_info.name}",True)

In [0]:
df_orders = spark.sql(f"SELECT * FROM {catalog}.bronze.staging_{data_source}")
df_orders.show(2)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FDEC83622503|Monday, December ...|     789622|  25891302|     39.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|Monday, December ...|     789622|  25891301|     26.0|2026-03-04 12:36:...|orders_2025_12_01...|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
only showing top 2 rows


In [0]:
df_orders = df_orders.filter(F.col("order_qty").isNotNull())
df_orders.show(10)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FDEC83622503|Monday, December ...|     789622|  25891302|     39.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|Monday, December ...|     789622|  25891301|     26.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|Monday, December ...|     789622|  25891503|    205.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|Monday, December ...|     789622|  25891501|    184.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|Monday, December ...|     789622|  25891101|    327.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|Monday, December ...|    INVALID|  25891401|    480.0|2026-03-04 12:36:...|orders_2025_12_01...|
|

In [0]:
df_orders = df_orders.withColumn("customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string"))
df_orders.show(10)    

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FDEC83622503|Monday, December ...|     789622|  25891302|     39.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|Monday, December ...|     789622|  25891301|     26.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|Monday, December ...|     789622|  25891503|    205.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|Monday, December ...|     789622|  25891501|    184.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|Monday, December ...|     789622|  25891101|    327.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|Monday, December ...|     999999|  25891401|    480.0|2026-03-04 12:36:...|orders_2025_12_01...|
|

In [0]:
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

#Format date after

df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)
df_orders.show(10)


+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FDEC83622503|          2025-12-01|     789622|  25891302|     39.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|          2025-12-01|     789622|  25891301|     26.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|          2025-12-01|     789622|  25891503|    205.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|          2025-12-01|     789622|  25891501|    184.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|          2025-12-01|     789622|  25891101|    327.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|          2025-12-01|     999999|  25891401|    480.0|2026-03-04 12:36:...|orders_2025_12_01...|
|

In [0]:
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

#cast product id
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))
df_orders.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FDEC83622503|          2025-12-01|     789622|  25891302|     39.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|          2025-12-01|     789622|  25891301|     26.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|          2025-12-01|     789622|  25891503|    205.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|          2025-12-01|     789622|  25891501|    184.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|          2025-12-01|     789622|  25891101|    327.0|2026-03-04 12:36:...|orders_2025_12_01...|
|FDEC83622503|          2025-12-01|     999999|  25891401|    480.0|2026-03-04 12:36:...|orders_2025_12_01...|
|

In [0]:
df_products = spark.table("consumer_goods.silver.products")
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

df_joined.show(5)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|        product_code|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+--------------------+
|FDEC84102502|          2025-12-01|     789102|  25891101|    311.0|2026-03-04 12:36:...|orders_2025_12_01...|e91ba9d665f90254d...|
|FDEC84220602|          2025-12-01|     789220|  25891103|    347.0|2026-03-04 12:36:...|orders_2025_12_01...|102628255d24304d6...|
|FDEC83503503|          2025-12-01|     789503|  25891503|    127.0|2026-03-04 12:36:...|orders_2025_12_01...|062f5574bbdf4386b...|
|FDEC84503602|          2025-12-01|     789503|  25891602|    193.0|2026-03-04 12:36:...|orders_2025_12_01...|778c2a7aa27bfdb21...|
|FDEC82321502|          2025-12-01|     789321|  25891201|    322.0|2026-03-

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

Staging table only for the increment data

In [0]:
# stagging for incremental data

df_joined.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.silver.staging_{data_source}")

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {catalog}.silver.staging_{data_source};")

df_gold.show(2)

+------------+----------+-------------+--------------------+----------+-------------+
|    order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+------------+----------+-------------+--------------------+----------+-------------+
|FDEC84102502|2025-12-01|       789102|e91ba9d665f90254d...|  25891101|        311.0|
|FDEC84220602|2025-12-01|       789220|102628255d24304d6...|  25891103|        347.0|
+------------+----------+-------------+--------------------+----------+-------------+
only showing top 2 rows


In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

Child data is daily level where as parent data is at monthly


In [0]:
df_child =  spark.sql(f"SELECT order_placement_date as date FROM {catalog}.silver.staging_{data_source}")

incremental_month_df = df_child.select(
    F.trunc("date", "MM").alias("start_month")
).distinct()

incremental_month_df.show()

incremental_month_df.createOrReplaceTempView("incremental_months")

+-----------+
|start_month|
+-----------+
| 2025-12-01|
+-----------+



In [0]:
monthly_table = spark.sql(f"""
    SELECT date, product_code, customer_code, sold_quantity
    FROM {catalog}.gold.sb_fact_orders sbf
    INNER JOIN incremental_months m
        ON trunc(sbf.date, 'MM') = m.start_month
""")

print("Total Rows: ", monthly_table.count())
monthly_table.show(10)

Total Rows:  1313
+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-12-01|e91ba9d665f90254d...|       789102|        311.0|
|2025-12-01|102628255d24304d6...|       789220|        347.0|
|2025-12-01|062f5574bbdf4386b...|       789503|        127.0|
|2025-12-01|778c2a7aa27bfdb21...|       789503|        193.0|
|2025-12-01|2e387cef1424d6e7b...|       789321|        322.0|
|2025-12-01|2e387cef1424d6e7b...|       789601|        352.0|
|2025-12-01|102628255d24304d6...|       789522|        397.0|
|2025-12-01|e91ba9d665f90254d...|       789721|        321.0|
|2025-12-01|2e387cef1424d6e7b...|       999999|        458.0|
|2025-12-01|102628255d24304d6...|       789220|        482.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


In [0]:
monthly_table.select('date').distinct().orderBy('date').show()

+----------+
|      date|
+----------+
|2025-12-01|
|2025-12-02|
|2025-12-03|
|2025-12-04|
|2025-12-05|
+----------+



In [0]:
df_monthly_recalc = (
    monthly_table
    .withColumn("month_start", F.trunc("date", "MM"))
    .groupBy("month_start", "product_code", "customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
    .withColumnRenamed("month_start", "date")   # month_start → date = first of month
)

df_monthly_recalc.show(10, truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-12-01|0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|789202       |806.0        |
|2025-12-01|77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9|999999       |3173.0       |
|2025-12-01|102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268|789422       |794.0        |
|2025-12-01|c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f|789201       |162.0        |
|2025-12-01|e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843|789402       |1648.0       |
|2025-12-01|3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345|789303       |286.0        |
|2025-12-01|716fa4e54b7894c910180276e0535d49af

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.gold.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly_recalc.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
DESCRIBE HISTORY consumer_goods.gold.fact_orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-03-04T12:36:49.000Z,71320613334569,yvishnu224@gmail.com,MERGE,"Map(predicate -> [""(((date#15956 = date#15950) AND (product_code#15957 = product_code#15942)) AND (cast(customer_code#15958 as bigint) = cast(customer_code#15941 as bigint)))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1469607878050113),68df0be0-ce2a-42f5-8292-a82547dcc0a5,0304-123401-b719owr-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4273, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2929, materializeSourceTimeMs -> 577, numTargetRowsInserted -> 558, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1350, numTargetRowsUpdated -> 0, numOutputRows -> 558, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 558, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 951)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
2,2026-03-03T12:33:47.000Z,71320613334569,yvishnu224@gmail.com,MERGE,"Map(predicate -> [""(((date#18625 = date#18619) AND (product_code#18626 = product_code#18611)) AND (cast(customer_code#18627 as bigint) = cast(customer_code#18610 as bigint)))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1469607878050112),54744384-08fc-4bac-a81d-d3f9c32e17ad,0303-112818-wxc4dad7-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 14048, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2545, materializeSourceTimeMs -> 359, numTargetRowsInserted -> 3060, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1279, numTargetRowsUpdated -> 0, numOutputRows -> 3060, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3060, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 863)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-02T11:42:09.000Z,71320613334569,yvishnu224@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(903424024329876),1ae2dd3f-8701-4ff9-aa7d-a967e6342af2,0302-113816-lp5wn68f-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 132760, numDeletionVectorsRemoved -> 0, numOutputRows -> 93055, numOutputBytes -> 132760)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-02T11:39:56.000Z,71320613334569,yvishnu224@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(903424024329876),c6fb62cd-90ab-4010-9ec2-2d0817eff2f2,0302-113816-lp5wn68f-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 93055, numOutputBytes -> 132760)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


Removing staging tables

In [0]:
%sql
DROP TABLE consumer_goods.bronze.staging_orders;

In [0]:
%sql
DROP TABLE consumer_goods.silver.staging_orders;